# Apache Iceberg Features: Schema Evolution & Time Travel in PySpark

This notebook demonstrates key capabilities of Apache Iceberg tables using PySpark:
1. **Table Ingestion Inspection:** View raw metadata tables (snapshots, files, history).
2. **Time Travel:** Query historical snapshots using snapshot IDs and historical timestamps.
3. **Schema Evolution:** Modify columns (adding, renaming, dropping) on the fly without rewriting the table or losing old history.

## Step 1: Initialize Spark Session with Apache Iceberg Support

First, we build the Spark session. We configure the GCS Hadoop Catalog (`hadoop_prod`) pointing to our warehouse bucket.

In [ ]:
import os
from pyspark.sql import SparkSession

# REPLACE WITH YOUR GCS BUCKET NAME
WAREHOUSE_PATH = "gs://YOUR_STAGING_BUCKET/warehouse"

spark = SparkSession.builder \
    .appName("Apache-Iceberg-TimeTravel-SchemaEvolution") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_prod", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_prod.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_prod.warehouse", WAREHOUSE_PATH) \
    .getOrCreate()

print("Spark Session initialized with Iceberg support!")

## Step 2: Read and Inspect current Iceberg Data

Let's load the table written by our Apache Beam pipeline and display the schema and sample records.

In [ ]:
table_name = "hadoop_prod.sensor_db.aggregates"

# Load table
df = spark.read.table(table_name)

print(f"Total records in {table_name}: {df.count()}")
df.printSchema()
df.show(5, truncate=False)

## Step 3: Iceberg Table History and Metadata Inspection

Iceberg exposes system metadata tables directly via SQL. You can query `.history`, `.snapshots`, `.files`, or `.manifests` namespaces.

In [ ]:
# Query the snapshots table to view commit history
snapshots_df = spark.read.table(f"{table_name}.snapshots")
print("Available snapshots list:")
snapshots_df.select("committed_at", "snapshot_id", "parent_id", "operation").show(truncate=False)

## Step 4: Time Travel (Snapshot & Timestamp Queries)

Time travel allows querying data from any past commit point.

In [ ]:
# Fetch a list of snapshots
snapshots = snapshots_df.orderBy("committed_at").collect()

if len(snapshots) >= 2:
    first_snapshot_id = snapshots[0]["snapshot_id"]
    second_snapshot_id = snapshots[1]["snapshot_id"]
    
    # 1. Querying by Snapshot ID
    print(f"--- Loading Data from First Snapshot ID: {first_snapshot_id} ---")
    df_v1 = spark.read.option("snapshot-id", first_snapshot_id).table(table_name)
    df_v1.show(5)
    print(f"Count: {df_v1.count()}")

    print(f"\n--- Loading Data from Second Snapshot ID: {second_snapshot_id} ---")
    df_v2 = spark.read.option("snapshot-id", second_snapshot_id).table(table_name)
    df_v2.show(5)
    print(f"Count: {df_v2.count()}")
    
    # 2. Querying by Timestamp (as-of-timestamp expects epoch milliseconds)
    # Let's target the exact time of the second snapshot
    epoch_ms = int(snapshots[1]["committed_at"].timestamp() * 1000)
    print(f"\n--- Loading Data as of epoch millisecond: {epoch_ms} ---")
    df_ts = spark.read.option("as-of-timestamp", epoch_ms).table(table_name)
    df_ts.show(5)
else:
    print("To show time travel, keep the Beam pipeline running to generate multiple window commits (snapshots).")

## Step 5: Schema Evolution

Iceberg supports full schema changes (in-place column additions, renaming, types, etc.) as metadata-only actions. Old data files are not modified.

### A. Add Column
Let's add a new string column `operator_notes` to our table.

In [ ]:
print("Adding new column 'operator_notes'...")
spark.sql(f"ALTER TABLE {table_name} ADD COLUMN operator_notes string")

print("Updated Schema:")
spark.read.table(table_name).printSchema()

Notice that existing records now successfully resolve the schema by returning `null` for the new field:

In [ ]:
spark.read.table(table_name).select("device_id", "avg_temperature", "operator_notes").show(5)

### B. Rename Column
Let's rename the column `error_count` to `fault_count`.

In [ ]:
print("Renaming 'error_count' to 'fault_count'...")
spark.sql(f"ALTER TABLE {table_name} RENAME COLUMN error_count TO fault_count")

print("Schema showing RENAME:")
spark.read.table(table_name).printSchema()
spark.read.table(table_name).select("device_id", "fault_count").show(5)

## Step 6: Shutdown Spark Session

In [ ]:
spark.stop()
print("Spark Session shut down.")